In [0]:
%run ../utils/adls_auth

In [0]:
# Disable deletion vectors for Synapse compatibility
spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")

In [0]:
from pyspark.sql.functions import col, sha2, when, date_trunc
from delta.tables import DeltaTable



SILVER_PATH = "abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/weather_silver"
GOLD_PATH = "abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_weather"

weather_df = spark.read.format("delta").load(SILVER_PATH)

dim_weather_df = (
    weather_df
    .withColumn("weather_hour", date_trunc("hour", col("weather_datetime")))
    .withColumn(
        "weather_category",
        when(col("weather_code").isin(0, 1), "clear")
        .when(col("weather_code").between(2, 3), "cloudy")
        .when(col("weather_code").between(51, 67), "rain")
        .when(col("weather_code").between(71, 77), "snow")
        .when(col("weather_code") >= 95, "storm")
        .otherwise("other"),
    )
    .withColumn("weather_key", sha2(col("weather_hour").cast("string"), 256))
    .dropDuplicates(["weather_hour"])
)

dim_weather_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PATH)
print(f"dim_weather built: {dim_weather_df.count()} rows.")